# 15a — Qwen3.5-0.8B size × limit × LoRA-1

**Runtime → Run all** on an **A100** (40GB or 80GB). No typing needed.

| | |
|---|---|
| Model | `unsloth/Qwen3.5-0.8B` (`qwen35_0_8b`) |
| Order | **SECOND (after 4B)** |
| Drive results | `MyDrive/stop-overthinking/results/0.8b/` |
| Ways | OFF · ON · limit 512/1024/2048/4096 · **LoRA-1 only** |
| Hour pot | ≤150h shared with the other size notebook |
| Test set | same **234** problems as the 2B thesis |

**Before first Run all:** copy this laptop repo once to Drive so Colab has the latest scripts:

```text
MyDrive/stop-overthinking/code/   ← full git repo (scripts/, data/, …)
```

DECISIONS #72.


## 1. Install packages

Unsloth (LoRA) + graders + flash-linear-attention (part of the fast path).


In [ ]:
%%capture
!pip install -q --upgrade unsloth
!pip install -q evalplus datasets flash-linear-attention
print("pip done")


## 2. GPU, Drive, latest code, budget

Picks batch size from GPU memory. **80GB → larger batch. 40GB → safer batch.**
Loads code from Drive mirror if present (recommended), else git clone.
**No input()** — safe for Run all.


In [ ]:
import os, sys, json, shutil, subprocess, time, glob
from IPython import get_ipython

name, mem = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip().split(", ")
GPU, GPU_GB = name, float(mem) / 1024
PRODUCTION_OK = ("A100" in GPU or "H100" in GPU) and GPU_GB > 35

# Batch: big enough to be fast, small enough to avoid OOM-split storms on LCB.
# 4B needs more memory per sequence than 0.8B / 2B.
MODEL_PROFILE = "qwen35_0_8b"
if GPU_GB >= 70:          # A100 80GB
    BATCH = 64 if MODEL_PROFILE == "qwen35_4b" else 128
elif GPU_GB > 35:         # A100 40GB
    BATCH = 32 if MODEL_PROFILE == "qwen35_4b" else 64
else:
    BATCH = 16
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) · batch {BATCH} · "
      + ("OK for the real run" if PRODUCTION_OK else "NOT an A100/H100 — stop"))

def rows(path):
    return sum(1 for l in open(path) if l.strip()) if os.path.exists(path) else 0

def sh(cmd):
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"FAILED: {cmd[:140]}")

from google.colab import drive
drive.mount("/content/drive")

D = "/content/drive/MyDrive/stop-overthinking/results"
WHEELS = "/content/drive/MyDrive/stop-overthinking/wheels"
DRIVE_CODE = "/content/drive/MyDrive/stop-overthinking/code"
os.makedirs(D, exist_ok=True)
os.makedirs(WHEELS, exist_ok=True)

# Prefer Drive copy of the repo (has your latest scripts). Fallback: GitHub.
if os.path.isdir(f"{DRIVE_CODE}/scripts"):
    print("Using Drive code mirror:", DRIVE_CODE)
    sh(f"rm -rf /content/thesis && mkdir -p /content/thesis")
    sh(f"cp -a {DRIVE_CODE}/. /content/thesis/")
else:
    print("WARNING: no Drive mirror at", DRIVE_CODE)
    print("  → cloning GitHub (may be older). For latest: upload this repo to that folder.")
    REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
    if not os.path.isdir("/content/thesis"):
        sh(f"git clone -q {REPO} /content/thesis")
    else:
        sh("cd /content/thesis && git pull -q || true")

os.chdir("/content/thesis")
sys.path.insert(0, "/content/thesis/scripts")
assert os.path.exists("scripts/gen_colab.py"), "scripts/gen_colab.py missing — fix code sync"
assert os.path.exists("scripts/hours_budget.py"), "scripts/hours_budget.py missing — fix code sync"

RUN_TAG = "0.8b"
MODEL = "qwen35_0_8b"
T = f"{D}/0.8b/raw"
SUMMARY = f"{D}/0.8b/SUMMARY.md"
HOURS = f"{D}/shared/hours_budget.json"
LORA1 = f"{T}/lora/lora1"
os.makedirs(T, exist_ok=True)
os.makedirs(LORA1, exist_ok=True)
os.makedirs(os.path.dirname(HOURS), exist_ok=True)

if not os.path.exists(HOURS):
    json.dump(dict(cap_hours=150, used_hours=0.0, runs={},
                   note="shared pot for 0.8B + 4B; keep ≥50h of 200 as buffer"),
              open(HOURS, "w"), indent=2)

MAXTOK = {"he": 4096, "lcb": 8192}
LIMITS = [512, 1024, 2048, 4096]
SRC = {"he": "humaneval", "lcb": "lcb"}
N = {"he": 164, "lcb": 70}
WAYS = {"off": ("thinking_off", None), "on": ("thinking_on", None)}
for L in LIMITS:
    WAYS[f"limit{L}"] = ("limit", None)
WAYS["lora1"] = ("thinking_on", LORA1)
COST_H = {"smoke": 0.5, "A": 18.0, "B1": 10.0, "B2": 1.5, "B3": 4.0, "C": 22.0}

import hours_budget as hb
print(f"model={MODEL} · out={T}")
print(f"shared hours left: {hb.left(HOURS):.1f} / 150")


## 3. Fast path (REQUIRED)

Without `causal-conv1d` + `fla`, Qwen3.5 runs at ~40 tok/s and burns the hour pot.
With them (2B thesis) we saw hundreds–1000+ tok/s.
This cell **stops the notebook** if the fast path is still off — do not skip it.


In [ ]:
import glob as _glob

def fast_path_ok():
    return subprocess.run(
        [sys.executable, "-c", "import causal_conv1d, fla"],
        capture_output=True).returncode == 0

for attempt in (1, 2):
    if fast_path_ok():
        break
    whl = _glob.glob(f"{WHEELS}/causal_conv1d*.whl")
    if not whl:
        print("building causal-conv1d once (~5–15 min) → saved on Drive for next time")
        sh(f"pip wheel -q causal-conv1d --no-build-isolation --no-deps -w {WHEELS}")
        whl = _glob.glob(f"{WHEELS}/causal_conv1d*.whl")
    if not whl:
        raise RuntimeError("pip wheel produced no causal_conv1d*.whl — see errors above")
    sh(f"pip install -q --force-reinstall {whl[0]}")
    sh("pip install -q --upgrade flash-linear-attention")
    if not fast_path_ok():
        print("wheel did not import — deleting and rebuilding once")
        for w in whl:
            try: os.remove(w)
            except OSError: pass

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda, "py", sys.version.split()[0])
if not fast_path_ok():
    raise RuntimeError(
        "FAST PATH OFF. Do not Run all further. "
        "Paste the torch/cuda/py line above into the chat.")
print("FAST PATH ON ✓")


## 4. Problems + overlap check


In [ ]:
for c in ["python scripts/test_prompts.py",
           "python scripts/build_problem_set.py",
           "python scripts/mbpp_data.py",
           "python scripts/overlap_check.py"]:
    print(">", c); sh(c)
sh(f"cp -f data/overlap_exclude.json {T}/overlap-exclude.json")
print("problems ready")


## 5. Helpers (resume-safe)


In [ ]:
BG = []

def ans_path(way, ds, folder=None, prefix="test"):
    return f"{folder or T}/{prefix}-{way}-{ds}.jsonl"

def answer(way, ds, tries, folder=None, n=0, maxtok=None, think_budget=None,
           only_ids=None, prefix="test", stop_on_repeat=True, problems="data/problems.json"):
    out = ans_path(way, ds, folder, prefix)
    want = (len(json.load(open(only_ids))) if only_ids else (n or N[ds])) * tries
    if rows(out) >= want:
        print(f"  {way:<10}{ds:<4} complete ({rows(out)}) - skipped"); return out
    policy, adapter = WAYS[way]
    if way.startswith("limit") and think_budget is None:
        think_budget = int(way.replace("limit", ""))
    cmd = (f'python scripts/gen_colab.py --policy {policy} --label {way} --model {MODEL} '
           f'--problems {problems} --source {SRC[ds]} --samples {tries} '
           f'--dtype auto --batch {BATCH} --max-tokens {maxtok or MAXTOK[ds]} '
           f'--think-budget {think_budget or 1024} --out "{out}"')
    if stop_on_repeat:
        cmd += " --stop-on-repeat"
    if n:
        cmd += f" --n {n}"
    if adapter:
        if not os.path.exists(f"{adapter}/adapter_config.json"):
            raise RuntimeError(f"LoRA missing at {adapter}")
        cmd += f' --adapter "{adapter}"'
    if only_ids:
        cmd += f' --only-ids "{only_ids}"'
    print(f"  {way:<10}{ds:<4} answering ...", flush=True)
    sh(cmd)
    return out

def graded_ok(ans):
    g = ans.replace(".jsonl", "-graded.csv")
    return (os.path.exists(g) and rows(g) - 1 == rows(ans)
            and os.path.getmtime(g) >= os.path.getmtime(ans))

def grade(ans, ds, problems="data/problems.json", wait=False):
    for a, p in BG:
        if a == ans:
            p.wait()
    if graded_ok(ans):
        return
    if ds == "mbpp":
        script = "grade_mbpp.py"
    elif ds == "he":
        script = "grade_plus.py"
    else:
        script = "grade_lcb.py"
    p = subprocess.Popen(
        [sys.executable, f"scripts/{script}", "--answers", ans, "--problems", problems],
        stdout=open(ans + ".grade.log", "w"), stderr=subprocess.STDOUT)
    if wait:
        p.wait(); print("  graded", ans)
    else:
        BG.append((ans, p))

def wait_grading():
    for ans, p in BG:
        p.wait(); print("  graded", ans)
    BG.clear()

def free_ways():
    return ["off", "on"] + [f"limit{L}" for L in LIMITS]

def stage(name, ways, tries, cost_key=None):
    todo = [(w, ds) for w in ways for ds in SRC if rows(ans_path(w, ds)) < N[ds] * tries]
    if not todo:
        print(f"stage {name}: already complete"); return
    assert PRODUCTION_OK, "Need A100/H100 with >35GB for production stages"
    cost = COST_H[cost_key or name]
    if not hb.gate(HOURS, RUN_TAG, name, cost):
        return
    hb.start_stage(HOURS, RUN_TAG, name)
    try:
        for w, ds in todo:
            grade(answer(w, ds, tries), ds)
    finally:
        hb.end_stage(HOURS, RUN_TAG, name)

print("helpers ready · free ways:", free_ways(), "· batch", BATCH)


## 6. Smoke test

Tiny check: every free way answers, counts thinking, grades. Then we know Run all is safe.


In [ ]:
SM = f"{T}/smoke"
if os.path.exists(f"{SM}/PASSED"):
    print("smoke already passed - skipped")
elif hb.gate(HOURS, RUN_TAG, "smoke", COST_H["smoke"]):
    hb.start_stage(HOURS, RUN_TAG, "smoke")
    try:
        shutil.rmtree(SM, ignore_errors=True); os.makedirs(SM)
        for w in free_ways():
            for ds in SRC:
                grade(answer(w, ds, 1, folder=SM, n=2, maxtok=256, think_budget=128),
                      ds, wait=True)
        for w in free_ways():
            for ds in SRC:
                r = [json.loads(l) for l in open(ans_path(w, ds, SM))]
                assert len(r) == 2, f"{w} {ds}: expected 2"
                if w == "off":
                    assert all(x["thinking_tokens"] == 0 for x in r), "OFF has thinking"
                else:
                    assert any(x["thinking_tokens"] > 0 for x in r), f"{w} 0 thinking"
                assert graded_ok(ans_path(w, ds, SM)), f"{w} {ds} not graded"
        open(f"{SM}/PASSED", "w").write("ok")
        print("SMOKE PASSED ✓")
    finally:
        hb.end_stage(HOURS, RUN_TAG, "smoke")


## 7. Stage A — free ways on 234 (try 1)

OFF · ON · limits 512/1024/2048/4096. Skips files already on Drive.


In [ ]:
assert PRODUCTION_OK
stage("A", free_ways(), tries=1, cost_key="A")
wait_grading()
print("stage A done. Hours left:", round(hb.left(HOURS), 1))


## 8. Stage B — LoRA-1 (full MBPP+ train, all kept examples)

No LoRA-2. Resume-safe.


In [ ]:
TR_ANS = f"{T}/train-mbpp-on.jsonl"
TRAIN = f"{T}/train-set-lora1.jsonl"
EXCLUDE = f"{T}/overlap-exclude.json"

if hb.gate(HOURS, RUN_TAG, "B1", COST_H["B1"]):
    hb.start_stage(HOURS, RUN_TAG, "B1")
    try:
        n_train = sum(1 for p in json.load(open("data/mbpp.json"))["problems"]
                      if p.get("split") == "train")
        want = n_train * 4
        print(f"MBPP+ train: {n_train} problems · want {want} answers")
        if rows(TR_ANS) < want:
            cmd = (f'python scripts/gen_colab.py --policy thinking_on --label on --model {MODEL} '
                   f'--problems data/mbpp.json --source mbpp --split train --samples 4 '
                   f'--dtype auto --batch {BATCH} --max-tokens 4096 --stop-on-repeat '
                   f'--out "{TR_ANS}"')
            print(cmd); sh(cmd)
        if not graded_ok(TR_ANS):
            sh(f'python scripts/grade_mbpp.py --answers "{TR_ANS}" --problems data/mbpp.json')
    finally:
        hb.end_stage(HOURS, RUN_TAG, "B1")

if os.path.exists(f"{LORA1}/adapter_config.json"):
    print("B2: LoRA-1 already trained - skipped")
elif hb.gate(HOURS, RUN_TAG, "B2", COST_H["B2"]):
    hb.start_stage(HOURS, RUN_TAG, "B2")
    try:
        sh(f'python scripts/make_train_set.py --answers "{TR_ANS}" --problems data/mbpp.json '
           f'--splits train --exclude "{EXCLUDE}" --out "{TRAIN}"')
        sh(f'python scripts/train_lora.py --train "{TRAIN}" --fraction 1.0 --epochs 3 '
           f'--model {MODEL} --out "{LORA1}"')
        assert os.path.exists(f"{LORA1}/adapter_config.json"), "LoRA-1 save failed"
    finally:
        hb.end_stage(HOURS, RUN_TAG, "B2")

if os.path.exists(f"{LORA1}/adapter_config.json"):
    stage("B3", ["lora1"], tries=1, cost_key="B3")
    wait_grading()
else:
    print("B3 skipped: no LoRA-1")


## 9. Stage C — second try (if hours remain)

Skipped automatically when the shared 150h pot is too low.


In [ ]:
ways_c = free_ways() + (["lora1"] if os.path.exists(f"{LORA1}/adapter_config.json") else [])
stage("C", ways_c, tries=2, cost_key="C")
wait_grading()
print("hours left:", round(hb.left(HOURS), 1))


## 10. Finish + SUMMARY.md on Drive


In [ ]:
wait_grading()
for way in free_ways() + (["lora1"] if os.path.exists(f"{LORA1}/adapter_config.json") else []):
    for ds in SRC:
        a = ans_path(way, ds)
        if rows(a) and not graded_ok(a):
            grade(a, ds, wait=True)

sh(f'python scripts/make_size_summary.py --dir "{T}" --run {RUN_TAG} --summary "{SUMMARY}"')
print("=== hours ===")
print(open(HOURS).read())
print(f"DONE ✓  results in Drive: results/{RUN_TAG}/")
